In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, accuracy_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# 1. Datan lataaminen
df = pd.read_csv('cicids2017_cleaned.csv')

# Valitaan 10 ominaisuutta
selected_features = [
    'Init_Win_bytes_forward', 'Flow IAT Mean', 'Packet Length Std', 'Subflow Fwd Bytes',
    'Flow Duration', 'Bwd Packet Length Mean', 'Total Length of Fwd Packets',
    'PSH Flag Count', 'Flow Packets/s', 'Destination Port'
]
label = 'Attack Type'

# 2. Mallien määrittely
# Määritellään vain muuttuvat osat: algoritmi ja sen omat parametrit
# SVM mallin kouluttaminen 20 000 rivin aineistolla kesti 3 minuuttia.
# Ajanpuutteen vuoksi rivejä ei lisätty, vaikka se todennäköisesti parantaisi mallin suorituskykyä.
models = {
    'SVM': (20000, SVC(cache_size=2000), [
        {   # SMOTE päällä
            'pca': [PCA(n_components=8, random_state=42)],
            'smote': [SMOTE(random_state=42)],
            'smote__k_neighbors': [1, 3],
            'model__C': [1, 10],
            'model__kernel': ['rbf']
        },
        {   # SMOTE pois
            'pca': [PCA(n_components=8, random_state=42)],
            'smote': ['passthrough'],
            'model__C': [1, 10],
            'model__kernel': ['rbf']
        }
    ]),
    'Naive Bayes': (100000, GaussianNB(), [
        {   # SMOTE päällä
            'pca': [PCA(n_components=5, random_state=42)],
            'smote': [SMOTE(random_state=42)],
            'smote__k_neighbors': [1],
            'model__var_smoothing': [1e-9, 1e-8]
        },
        {   # SMOTE pois
            'pca': [PCA(n_components=5, random_state=42)],
            'smote': ['passthrough'],
            'model__var_smoothing': [1e-9, 1e-8]
        }
    ])
}

# 3. Mallien harjoittaminen silmukassa
for name, (size, algo, params) in models.items():
    print(f"\n--- Ajetaan {name} (otos: {size}) ---")

    # Otetaan otos ja siivotaan harvinaiset luokat
    df_s = df[selected_features + [label]].sample(n=size, random_state=42)
    v_counts = df_s[label].value_counts()
    df_s = df_s[df_s[label].isin(v_counts[v_counts >= 15].index)]

    X = df_s.drop(label, axis=1)
    y = df_s[label]

    # Jako 60/20/20
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
    _, X_test, _, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

    # Rakennetaan putki
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(random_state=42)),
        ('smote', SMOTE(random_state=42)),
        ('model', algo)
    ])

    # Suoritetaan haku
    grid = GridSearchCV(pipe, params, cv=3, n_jobs=-1, error_score=0, verbose=1)
    grid.fit(X_train, y_train)

    # Tulostetaan kaikkien kokeiltujen yhdistelmien tarkkuudet
    print(f"\nKaikkien kokeiltujen {name}-mallien tulokset (Top 5):")
    results = pd.DataFrame(grid.cv_results_)

    # Luodaan selkeämpi sarake SMOTE-tiedolle
    results['SMOTE_status'] = results['param_smote'].apply(lambda x: 'Päällä' if x != 'passthrough' else 'Pois')

    # Järjestetään ja valitaan tärkeimmät sarakkeet näkyviin
    results = results.sort_values(by='mean_test_score', ascending=False)
    print(results[['mean_test_score', 'SMOTE_status', 'params']].head(10))

    # Parhaan mallin lopulliset tulokset testidatalla
    y_pred = grid.predict(X_test)
    print(f"\n--- PARAS {name.upper()} ---")
    print(f"parhaat parametrit: {grid.best_params_}")
    print(f"tarkkuus testidatalla: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred, zero_division=0))


--- Ajetaan SVM (otos: 20000) ---
Fitting 3 folds for each of 6 candidates, totalling 18 fits

Kaikkien kokeiltujen SVM-mallien tulokset (Top 10):
   mean_test_score SMOTE_status  \
5         0.957222         Pois   
4         0.942545         Pois   
2         0.873750       Päällä   
3         0.873333       Päällä   
1         0.827724       Päällä   
0         0.820302       Päällä   

                                              params  
5  {'model__C': 10, 'model__kernel': 'rbf', 'pca'...  
4  {'model__C': 1, 'model__kernel': 'rbf', 'pca':...  
2  {'model__C': 10, 'model__kernel': 'rbf', 'pca'...  
3  {'model__C': 10, 'model__kernel': 'rbf', 'pca'...  
1  {'model__C': 1, 'model__kernel': 'rbf', 'pca':...  
0  {'model__C': 1, 'model__kernel': 'rbf', 'pca':...  

--- PARAS SVM ---
parhaat parametrit: {'model__C': 10, 'model__kernel': 'rbf', 'pca': PCA(n_components=8, random_state=42), 'smote': 'passthrough'}
tarkkuus testidatalla: 0.9530
                precision    recall  f1-sc